#### Objetivo

- Esse notebook consiste em concatenar os dados dos eventos de todas as partidas das competições entre todas as suas temporadas. 

    A ideia é de analisar principalmente a disponibilidade dos dados de tracking e entender qual/quais temporadas sugerem estar mais consistentes de serem utilizadas.

In [88]:
import pandas as pd
from pathlib import Path
import os

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql import functions as F
from pyspark.sql.types import *

pd.set_option('display.max_columns', None)

In [89]:
competitions = [competition for competition in os.listdir(str(Path().resolve().parent.parent / "data" / "events"))]
competitions

['1', '42']

In [90]:
competitions_seasons = {
    competitions[id]: [season for season in os.listdir(str(Path().resolve().parent.parent / "data" / "events" / competitions[id]))]
    for id in range(len(competitions))
}
competitions_seasons

{'1': ['2020-2021', '2021-2022', '2022-2023', '2023-2024', '2024-2025'],
 '42': ['2023', '2024', '2025']}

In [91]:
def get_season_events_parquet_file_paths(events_competition_season_folder_path):
    
    season_events_parquet_file_paths = [
        str(Path(events_competition_season_folder_path) / season_event_parquet_file) 
        for season_event_parquet_file in os.listdir(events_competition_season_folder_path) 
        if season_event_parquet_file.endswith('.parquet')
        ]
    
    return season_events_parquet_file_paths

In [92]:
def get_competition_seasons_parquet_file_paths(competition, seasons):

    competition_seasons_parquet_file_paths = []

    for season in seasons:

        events_competition_season_folder_path = str(Path().resolve().parent.parent / "data" / "events" / competition / season)

        competition_seasons_parquet_file_paths.extend(get_season_events_parquet_file_paths(events_competition_season_folder_path))
    
    return competition_seasons_parquet_file_paths

In [93]:
competitions_seasons_events_parquet_file_paths = []

for competition, seasons in competitions_seasons.items():

    competitions_seasons_events_parquet_file_paths.extend(get_competition_seasons_parquet_file_paths(competition, seasons))

competitions_seasons_events_parquet_file_paths

['C:\\Users\\Matheus\\Documents\\Mestrado\\Dissertação\\defensive-performance-prediction\\data\\events\\1\\2020-2021\\events_1_2020-2021_part1.parquet',
 'C:\\Users\\Matheus\\Documents\\Mestrado\\Dissertação\\defensive-performance-prediction\\data\\events\\1\\2020-2021\\events_1_2020-2021_part2.parquet',
 'C:\\Users\\Matheus\\Documents\\Mestrado\\Dissertação\\defensive-performance-prediction\\data\\events\\1\\2021-2022\\events_1_2021-2022_part1.parquet',
 'C:\\Users\\Matheus\\Documents\\Mestrado\\Dissertação\\defensive-performance-prediction\\data\\events\\1\\2021-2022\\events_1_2021-2022_part2.parquet',
 'C:\\Users\\Matheus\\Documents\\Mestrado\\Dissertação\\defensive-performance-prediction\\data\\events\\1\\2022-2023\\events_1_2022-2023_part1.parquet',
 'C:\\Users\\Matheus\\Documents\\Mestrado\\Dissertação\\defensive-performance-prediction\\data\\events\\1\\2022-2023\\events_1_2022-2023_part2.parquet',
 'C:\\Users\\Matheus\\Documents\\Mestrado\\Dissertação\\defensive-performance-pred

In [94]:
# Criação da sessão Spark local
spark = SparkSession.builder.master("local[*]").appName("events").getOrCreate()

# Criação do dataframe concatenando todos os arquivos parquet dos eventos das partidas entre as temporadas de todas as competições
df = spark.read.parquet(*competitions_seasons_events_parquet_file_paths)

df.show(5)

+--------------------+-------------+------+---------+------+-----------------+------------+--------------------+--------------+-----------------------+--------+--------------------+--------------------+--------------------+--------------------+---------+-----------+-------+------------+
|                  id|competitionId|gameId|   season|period|periodDescription|   eventType|eventTypeDescription|startGameClock|startFormattedGameClock|homeTeam|             details|         homePlayers|         awayPlayers|               balls|player.id|player.name|team.id|   team.name|
+--------------------+-------------+------+---------+------+-----------------+------------+--------------------+--------------+-----------------------+--------+--------------------+--------------------+--------------------+--------------------+---------+-----------+-------+------------+
|4a60f04fd9c90fe77...|            1|   152|2020-2021|     1|       First half|FIRSTKICKOFF| First half kick off|             0|         

In [95]:
df.printSchema()

root
 |-- id: string (nullable = true)
 |-- competitionId: long (nullable = true)
 |-- gameId: long (nullable = true)
 |-- season: string (nullable = true)
 |-- period: long (nullable = true)
 |-- periodDescription: string (nullable = true)
 |-- eventType: string (nullable = true)
 |-- eventTypeDescription: string (nullable = true)
 |-- startGameClock: long (nullable = true)
 |-- startFormattedGameClock: string (nullable = true)
 |-- homeTeam: boolean (nullable = true)
 |-- details: string (nullable = true)
 |-- homePlayers: string (nullable = true)
 |-- awayPlayers: string (nullable = true)
 |-- balls: string (nullable = true)
 |-- player.id: long (nullable = true)
 |-- player.name: string (nullable = true)
 |-- team.id: long (nullable = true)
 |-- team.name: string (nullable = true)



In [96]:
df.select('homePlayers').first()

Row(homePlayers='[{"speed": null, "y": -10.55, "x": 10.919, "player": {"id": 37, "name": "Virgil van Dijk"}, "visibility": "VISIBLE", "confidence": "HIGH", "jerseyNum": null}, {"speed": null, "y": -12.984, "x": 4.607, "player": {"id": 45, "name": "Georginio Wijnaldum"}, "visibility": "VISIBLE", "confidence": "HIGH", "jerseyNum": null}, {"speed": null, "y": -29.974, "x": 0.049, "player": {"id": 54, "name": "Sadio Mané"}, "visibility": "VISIBLE", "confidence": "HIGH", "jerseyNum": null}, {"speed": null, "y": 18.107, "x": 1.143, "player": {"id": 56, "name": "Mohamed Salah"}, "visibility": "VISIBLE", "confidence": "HIGH", "jerseyNum": null}, {"speed": null, "y": -16.913, "x": 0.122, "player": {"id": 59, "name": "Roberto Firmino"}, "visibility": "VISIBLE", "confidence": "HIGH", "jerseyNum": null}, {"speed": null, "y": 0.44, "x": -0.319, "player": {"id": 46, "name": "Naby Keïta"}, "visibility": "VISIBLE", "confidence": "HIGH", "jerseyNum": null}, {"speed": null, "y": -1.327, "x": 35.382, "pl

In [97]:
# Schema em Pyspark para poder parsear o json dos dados de tracking que está como string
schema = ArrayType(
    StructType([
        StructField("x", FloatType(), True),
        StructField("y", FloatType(), True),
        StructField("player", StructType([
            StructField("id", IntegerType(), True), 
            StructField("name", StringType(), True)]), 
            True),
        StructField("visibility", StringType(), True),
        StructField("confidence", StringType(), True),
        StructField("jerseyNum", StringType(), True)      
    ])
)

In [98]:
df = df.withColumns({
    # Cria coluna com json parseado para Lista de dicionários para dados de tracking do time mandante
    "homePlayers_parsed": from_json("homePlayers", schema),
    
    # Cria coluna com json parseado para Lista de dicionários para dados de tracking do time adversário
    "awayPlayers_parsed": from_json("awayPlayers", schema)
})

In [99]:
df = df.withColumns({

    # Confere se nos dados de tracking do mandante não é lista vazia e tem pelo menos um x,y preenchidos entre os jogadores (True/False convertido para binário)
    "has_tracking_home":
    exists(
        col("homePlayers_parsed"),
        lambda k: k.x.isNotNull() & k.y.isNotNull()
    ).cast("int"),

    # Confere se nos dados de tracking do adversário não é lista vazia e tem pelo menos um x,y preenchidos entre os jogadores (True/False convertido para binário)
    "has_tracking_away":
    exists(
        col("awayPlayers_parsed"),
        lambda k: k.x.isNotNull() & k.y.isNotNull()
    ).cast("int"),

    # Confere se nos dados de tracking do mandante não é lista vazia e tem x,y preenchidos dos 11 jogadores (True/False convertido para binário)
    "all_tracking_home":
    ((size(col("homePlayers_parsed")) > 0) & 
    forall(
        col("homePlayers_parsed"),
        lambda k: k.x.isNotNull() & k.y.isNotNull()
    )).cast("int"),

    # Confere se nos dados de tracking do adversário não é lista vazia e tem x,y preenchidos dos 11 jogadores (True/False convertido para binário)
    "all_tracking_away":
    ((size(col("awayPlayers_parsed")) > 0) & 
    forall(
        col("awayPlayers_parsed"),
        lambda k: k.x.isNotNull() & k.y.isNotNull()
    )).cast("int"),

    # Traz a quantidade de dicionários de cada evento para saber se tem 11 jogadores do time mandante e adversário
    "len_tracking_home": size(col("homePlayers_parsed")),
    "len_tracking_away": size(col("awayPlayers_parsed"))
}
)

In [100]:
df.show(5)

+--------------------+-------------+------+---------+------+-----------------+------------+--------------------+--------------+-----------------------+--------+--------------------+--------------------+--------------------+--------------------+---------+-----------+-------+------------+--------------------+--------------------+-----------------+-----------------+-----------------+-----------------+-----------------+-----------------+
|                  id|competitionId|gameId|   season|period|periodDescription|   eventType|eventTypeDescription|startGameClock|startFormattedGameClock|homeTeam|             details|         homePlayers|         awayPlayers|               balls|player.id|player.name|team.id|   team.name|  homePlayers_parsed|  awayPlayers_parsed|has_tracking_home|has_tracking_away|all_tracking_home|all_tracking_away|len_tracking_home|len_tracking_away|
+--------------------+-------------+------+---------+------+-----------------+------------+--------------------+------------

In [101]:
df_agg_games = (
    df.groupBy("gameId")
      .agg(
          max("competitionId").alias("competitionId"),
          max("season").alias("season"),
          round(F.mean("has_tracking_home"), 3).alias("has_tracking_home_percent"),
          round(F.mean("has_tracking_away"), 3).alias("has_tracking_away_percent"),
          round(F.mean("all_tracking_home"), 3).alias("all_tracking_home_percent"),
          round(F.mean("all_tracking_away"), 3).alias("all_tracking_away_percent"),
          max("len_tracking_home").alias("tracking_home_len"),
          max("len_tracking_away").alias("tracking_away_len"),
      )
)

df_agg_games.cache()

DataFrame[gameId: bigint, competitionId: bigint, season: string, has_tracking_home_percent: double, has_tracking_away_percent: double, all_tracking_home_percent: double, all_tracking_away_percent: double, tracking_home_len: int, tracking_away_len: int]

In [102]:
df_agg_games.show(5)

+------+-------------+---------+-------------------------+-------------------------+-------------------------+-------------------------+-----------------+-----------------+
|gameId|competitionId|   season|has_tracking_home_percent|has_tracking_away_percent|all_tracking_home_percent|all_tracking_away_percent|tracking_home_len|tracking_away_len|
+------+-------------+---------+-------------------------+-------------------------+-------------------------+-------------------------+-----------------+-----------------+
|   474|            1|2020-2021|                    0.998|                    0.998|                    0.998|                    0.998|               11|               11|
|  4590|            1|2022-2023|                      1.0|                      1.0|                      1.0|                      1.0|               11|               11|
| 12568|           42|     2023|                      1.0|                      1.0|                      1.0|                      1.0

In [103]:
print('Quantidade de partidas sem nenhum dado de tracking do mandante:', df_agg_games.filter(col('has_tracking_home_percent') == 0).count())
print('Quantidade de partidas sem nenhum dado de tracking do adversário:', df_agg_games.filter(col('has_tracking_home_percent') == 0).count())
print('Quantidade de partidas com dados de tracking incomuns (diferente de 11) do mandante:', df_agg_games.filter(col('tracking_home_len') != 11).count())
print('Quantidade de partidas com dados de tracking incomuns (diferente de 11) do adversário:', df_agg_games.filter(col('tracking_away_len') != 11).count())
print('Quantidade de partidas que não possui dados de tracking dos 22 jogadores:', df_agg_games.filter((col('all_tracking_home_percent') < 1) | (col('has_tracking_away_percent') < 1)).count())

Quantidade de partidas sem nenhum dado de tracking do mandante: 52
Quantidade de partidas sem nenhum dado de tracking do adversário: 52
Quantidade de partidas com dados de tracking incomuns (diferente de 11) do mandante: 208
Quantidade de partidas com dados de tracking incomuns (diferente de 11) do adversário: 224
Quantidade de partidas que não possui dados de tracking dos 22 jogadores: 364


- Existem 52 partidas que não possuem nenhum dado de tracking para o time mandante ou adversário.
- Existem 208 partidas que possuem dados de tracking a mais ou a menos de jogadores do time mandante e 224 partidas do time adversário.
- Existem 364 partidas que não possuem dados de tracking de todos os 22 jogadores em campo.

In [104]:
df_agg_games.filter((col('has_tracking_home_percent') == 0) | (col('has_tracking_away_percent') == 0)).groupBy('competitionId', 'season').count().show()

+-------------+------+-----+
|competitionId|season|count|
+-------------+------+-----+
|           42|  2025|   52|
+-------------+------+-----+



- Dentre as 52 partidas com dados de tracking faltando, todas são da temporada de 2025 no Brasileirão.

In [105]:
df_agg_games.filter((col('tracking_home_len') != 11) | (col('tracking_home_len') != 11)).groupBy('competitionId', 'season').count().orderBy('competitionId', 'season').show()

+-------------+---------+-----+
|competitionId|   season|count|
+-------------+---------+-----+
|            1|2020-2021|   16|
|            1|2021-2022|   10|
|            1|2022-2023|    3|
|            1|2023-2024|    1|
|            1|2024-2025|   10|
|           42|     2023|   13|
|           42|     2024|   11|
|           42|     2025|  144|
+-------------+---------+-----+



- Apesar de existir registros faltando de tracking de jogadores em alguns eventos, são poucos registros de forma geral. 
- Porém, o ano de 2025 do Brasileirão apresenta uma quantidade elevada de 144 registros com ausência de dados de tracking de algum jogador.

In [106]:
df_agg_games.filter((col('has_tracking_home_percent') < 1) | (col('has_tracking_away_percent') < 1)).groupBy('competitionId', 'season').count().orderBy('competitionId', 'season').show()

+-------------+---------+-----+
|competitionId|   season|count|
+-------------+---------+-----+
|            1|2020-2021|  200|
|            1|2021-2022|    5|
|            1|2022-2023|    3|
|            1|2023-2024|   28|
|            1|2024-2025|   10|
|           42|     2023|   25|
|           42|     2024|   27|
|           42|     2025|   66|
+-------------+---------+-----+



- Pode-se notar que todas as temporadas das duas competições possuem uma certa quantidade de dados faltando para os 22 jogadores. 
- Tratando-se de Premier League, as temporadas com a menor quantidade de dados de tracking faltando seriam as de 2021-2022, 2022-2023 e 2024-2025.
- Já sobre o Brasileirão, as temporadas de 2023 e 2024 possuem uma quantidade um pouco elevada, mas seriam as mais adequadas para se trabalhar. A temporada de 2025 possui 109 eventos faltando informações de tracking, sendo que 52 delas faltam 100% dos dados de tracking.
- Para a análise de uma temporada em específico, é recomendado **começar pela de 2022-2023 da Premier League** por apresentar o menor número de registros faltantes.